# TP 5 – Sélection de variables (filtre et wrapper)

Dataset : Breast Cancer Wisconsin (Diagnostic). J'utilise `random_state=42`, un seul split 70/30 stratifié, une validation croisée à 5 plis sur le train, et je n'utilise le test qu'à la fin pour le tableau final (D.3).

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.feature_selection import (VarianceThreshold, SelectKBest, f_classif,
                                       mutual_info_classif, SequentialFeatureSelector,
                                       RFE, RFECV)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import BaseEstimator, TransformerMixin

warnings.filterwarnings("ignore")
SEED = 42
skf = StratifiedKFold(5, shuffle=True, random_state=SEED)

## Partie A – Données et baseline

In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame
print("dimensions :", df.shape)
print(df.dtypes.value_counts())
print("valeurs manquantes :", df.isna().sum().sum())
print("lignes dupliquees :", df.duplicated().sum())
print(df.describe().T.head(10))

Le dataset a 569 lignes et 30 variables numériques plus la cible. Il n'y a ni valeurs manquantes ni doublons.

In [ ]:
X = df.drop(columns="target")
y = df["target"]
print(y.value_counts())
print(y.value_counts(normalize=True).round(3))

plt.figure(figsize=(4, 3))
sns.countplot(x=y)
plt.xticks([0, 1], ["malin (0)", "benin (1)"])
plt.title("Distribution de target")
plt.tight_layout()
plt.show()

Il y a 62,7 % de benign et 37,3 % de malignant. Le DummyClassifier (toujours « benign ») a déjà 0,6257 d'accuracy en test, donc il faut regarder aussi le F1.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=SEED)
print(X_train.shape, X_test.shape)

dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
acc_dummy = accuracy_score(y_test, dummy.predict(X_test))
print("accuracy dummy (test) :", round(acc_dummy, 4))

scaler = StandardScaler().fit(X_train)
Xtr_s = scaler.transform(X_train)
Xte_s = scaler.transform(X_test)

In [ ]:
lr = LogisticRegression(max_iter=1000)
t0 = time.perf_counter()
lr.fit(Xtr_s, y_train)
t_base = time.perf_counter() - t0

p_tr = lr.predict(Xtr_s)
p_te = lr.predict(Xte_s)
print("train acc", round(accuracy_score(y_train, p_tr), 4), "F1", round(f1_score(y_train, p_tr), 4))
print("test  acc", round(accuracy_score(y_test, p_te), 4), "F1", round(f1_score(y_test, p_te), 4))
print("temps entrainement (s):", round(t_base, 4))

baseline = pd.DataFrame([{"Methode": "Baseline (30 features)", "Nb features": 30,
                          "Test acc": accuracy_score(y_test, p_te),
                          "Test F1": f1_score(y_test, p_te), "Temps (s)": t_base}])
print(baseline)

La baseline avec les 30 variables donne environ 0,987 en train et 0,988 en test (F1 = 0,991 en test). Pas de gros surapprentissage.

In [ ]:
corr = X_train.corr()
plt.figure(figsize=(13, 11))
sns.heatmap(corr, cmap="coolwarm", center=0, square=True, cbar_kws={"shrink": .6})
plt.title("Matrice de correlation (train)")
plt.tight_layout()
plt.show()

# paires tres correlees
paires = []
cols = list(corr.columns)
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        if abs(corr.iloc[i, j]) > 0.9:
            paires.append((cols[i], cols[j], round(corr.iloc[i, j], 3)))
print(len(paires), "paires avec |corr| > 0.9")
for p in sorted(paires, key=lambda t: -abs(t[2])):
    print(p)

Groupes de variables presque redondantes :
- radius, perimeter et area (en mean, worst et error), par exemple mean radius / mean perimeter = 0,998
- la version mean et la version worst de la même mesure (mean radius / worst radius = 0,974)
- concavity et concave points (0,917)

J'ai trouvé 23 paires avec une corrélation > 0,9.

Pourquoi ce dataset est un bon candidat pour la sélection :
- beaucoup de variables sont presque identiques, donc on peut en enlever sans perdre d'information
- on a seulement 398 observations d'entraînement pour 30 variables
- la baseline est déjà bonne (≈ 99 %), donc on voit bien ce que chaque méthode fait perdre

## Partie B – Méthodes filtre

### B.1 Variance threshold

In [ ]:
variances = X_train.var().sort_values()
print(variances.head(8))
rows = []
for th in [0, 0.01, 0.1]:
    vt = VarianceThreshold(threshold=th).fit(X_train)
    removed = list(X_train.columns[~vt.get_support()])
    rows.append({"seuil": th, "nb supprimees": len(removed), "supprimees": removed})
tab_b1 = pd.DataFrame(rows)
print(tab_b1.to_string())

Avec le seuil 0 aucune variable n'est supprimée, avec 0,01 on en supprime 16 et avec 0,1 on en supprime 19.

Après standardisation toutes les variances valent 1, donc le filtre ne peut plus rien distinguer (rien n'est supprimé). Ici il est peu utile parce que la variance dépend surtout de l'unité (l'aire est très grande, la dimension fractale très petite), pas de l'utilité de la variable. Par exemple avec 0,01 on enlève mean concave points, qui est pourtant l'une des variables les plus liées à la cible.

### B.2 Variables corrélées

In [ ]:
def remove_correlated(X, y, threshold):
    corr = X.corr().abs()
    cor_target = X.apply(lambda c: abs(np.corrcoef(c, y)[0, 1]))
    cols = list(X.columns)
    a_supprimer = set()
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            a, b = cols[i], cols[j]
            if a in a_supprimer or b in a_supprimer:
                continue
            if corr.loc[a, b] > threshold:
                # on garde celle qui est la plus correlee avec la cible
                if cor_target[a] >= cor_target[b]:
                    a_supprimer.add(b)
                else:
                    a_supprimer.add(a)
    return [c for c in cols if c not in a_supprimer]

rows = []
kept_corr = {}
for th in [0.95, 0.90, 0.80]:
    kept = remove_correlated(X_train, y_train, th)
    kept_corr[th] = kept
    pipe = Pipeline([("sc", StandardScaler()), ("lr", LogisticRegression(max_iter=1000))])
    pipe.fit(X_train[kept], y_train)
    acc = accuracy_score(y_test, pipe.predict(X_test[kept]))
    cv = cross_val_score(pipe, X_train[kept], y_train, cv=skf).mean()
    rows.append({"seuil": th, "nb restantes": len(kept), "test acc": round(acc, 4), "cv acc (train)": round(cv, 4)})
tab_b2 = pd.DataFrame(rows)
print(tab_b2)
print("baseline test acc :", round(baseline["Test acc"][0], 4),
      " cv baseline :", round(cross_val_score(Pipeline([("sc", StandardScaler()), ("lr", LogisticRegression(max_iter=1000))]), X_train, y_train, cv=skf).mean(), 4))
for th in kept_corr:
    print(th, kept_corr[th])

| seuil | variables restantes | accuracy test |
|---|---|---|
| 0,95 | 23 | 0,9825 |
| 0,90 | 19 | 0,9708 |
| 0,80 | 13 | 0,9415 |

La baseline a 0,9883. Enlever les variables redondantes ne suffit pas à améliorer l'accuracy : à 0,95 on perd 1 patient mais on garde 7 variables de moins, et plus le seuil baisse plus l'accuracy diminue, parce qu'on finit par enlever des variables utiles pour la cible.

### B.3 SelectKBest

In [ ]:
F, pval = f_classif(X_train, y_train)
mi = mutual_info_classif(X_train, y_train, random_state=SEED)
scores = pd.DataFrame({"F": F, "p-value": pval, "MI": mi}, index=X_train.columns)
print(scores.sort_values("F", ascending=False).head(10))

fig, ax = plt.subplots(1, 2, figsize=(14, 7))
scores["F"].sort_values().plot.barh(ax=ax[0], color="tab:blue")
ax[0].set_title("Score F (ANOVA)")
scores["MI"].sort_values().plot.barh(ax=ax[1], color="tab:orange")
ax[1].set_title("Information mutuelle")
plt.tight_layout()
plt.show()

rho, p_rho = spearmanr(scores["F"], scores["MI"])
print("Spearman F vs MI :", round(rho, 3), "p =", p_rho)

ks = [2, 5, 10, 15, 20, 25, 30]
def mi_func(X, y):
    return mutual_info_classif(X, y, random_state=SEED)
res_k = {"F": [], "MI": []}
std_k = {"F": [], "MI": []}
for nom, fonc in [("F", f_classif), ("MI", mi_func)]:
    for k in ks:
        pipe = Pipeline([("sc", StandardScaler()),
                         ("sel", SelectKBest(fonc, k=k)),
                         ("lr", LogisticRegression(max_iter=1000))])
        s = cross_val_score(pipe, X_train, y_train, cv=skf, scoring="accuracy")
        res_k[nom].append(s.mean())
        std_k[nom].append(s.std())

tab_k = pd.DataFrame({"k": ks,
                      "acc F": np.round(res_k["F"], 4), "std F": np.round(std_k["F"], 4),
                      "acc MI": np.round(res_k["MI"], 4), "std MI": np.round(std_k["MI"], 4)})
print(tab_k)

plt.figure(figsize=(6, 4))
for nom in ["F", "MI"]:
    plt.errorbar(ks, res_k[nom], yerr=std_k[nom], marker="o", capsize=3, label=nom)
plt.xlabel("k"); plt.ylabel("accuracy CV (5 plis)")
plt.title("SelectKBest : accuracy en fonction de k")
plt.legend(); plt.grid(alpha=.3)
plt.tight_layout()
plt.show()

# regle de l'ecart-type
best_k = {}
for nom in ["F", "MI"]:
    m = np.array(res_k[nom]); s = np.array(std_k[nom])
    ib = m.argmax()
    seuil = m[ib] - s[ib]
    best_k[nom] = min(k for k, mm in zip(ks, m) if mm >= seuil)
    print(nom, "meilleur k =", ks[ib], "acc =", round(m[ib], 4), "-> plus petit k dans 1 ecart-type :", best_k[nom])

# exemple de redondance : top 10 F
print("top 10 F :", list(scores["F"].sort_values(ascending=False).head(10).index))

Les deux classements (F et information mutuelle) sont très proches, la corrélation de Spearman est de 0,966. Les variables en haut sont worst concave points, worst perimeter, mean concave points, worst radius...

Avec la règle « plus petit k dans un écart-type du meilleur » je choisis k = 20 pour les deux scores (accuracy CV de 0,9874 pour F et 0,9799 pour MI).

Un filtre univarié garde des variables presque identiques parce qu'il note chaque variable séparément avec la cible. Radius, perimeter et area sont toutes liées à la cible donc elles ont toutes un bon score, et le filtre ne voit pas qu'elles disent la même chose.

## Partie C – Méthodes wrapper

### C.1 Forward selection

In [ ]:
noms = np.array(X_train.columns)

t0 = time.perf_counter()
sfs_f = SequentialFeatureSelector(LogisticRegression(max_iter=1000), n_features_to_select=10,
                                  direction="forward", scoring="accuracy", cv=skf)
sfs_f.fit(Xtr_s, y_train)
t_sfs_f = time.perf_counter() - t0
feat_sfs_f = list(noms[sfs_f.get_support()])
print("Forward sklearn (%.1f s) :" % t_sfs_f, feat_sfs_f)

In [ ]:
def forward_selection(Xa, ya, max_feat=10, eps=0.001):
    modele = LogisticRegression(max_iter=1000)
    choisies = []
    restantes = list(range(Xa.shape[1]))
    historique = []          # (nb features, score moyen, ecart-type, liste)
    score_prec = 0
    while len(choisies) < max_feat:
        meilleur_score, meilleur_f, meilleur_std = -1, None, None
        for f in restantes:
            cols = choisies + [f]
            s = cross_val_score(modele, Xa[:, cols], ya, cv=skf, scoring="accuracy")
            if s.mean() > meilleur_score:
                meilleur_score, meilleur_f, meilleur_std = s.mean(), f, s.std()
        gain = meilleur_score - score_prec
        if len(choisies) > 0 and gain < eps:
            break
        choisies.append(meilleur_f)
        restantes.remove(meilleur_f)
        score_prec = meilleur_score
        historique.append((len(choisies), meilleur_score, meilleur_std, list(choisies)))
    return historique

# version sans critere d'arret (pour comparer avec sklearn)
t0 = time.perf_counter()
hist_f_full = forward_selection(Xtr_s, y_train, max_feat=10, eps=-1)
t_forward_perso = time.perf_counter() - t0
feat_perso_f = [noms[i] for i in hist_f_full[-1][3]]
print("Forward perso (%.1f s) :" % t_forward_perso, feat_perso_f)
print("meme ensemble que sklearn ?", set(feat_perso_f) == set(feat_sfs_f))

# version avec epsilon = 0.001
hist_f = forward_selection(Xtr_s, y_train, max_feat=10, eps=0.001)
print("avec eps=0.001, arret a", len(hist_f), "features")
for h in hist_f_full:
    print(h[0], round(h[1], 4), "+", noms[h[3][-1]])

In [ ]:
plt.figure(figsize=(6, 4))
plt.errorbar([h[0] for h in hist_f_full], [h[1] for h in hist_f_full],
             yerr=[h[2] for h in hist_f_full], marker="o", capsize=3)
plt.axvline(len(hist_f), color="red", ls="--", label="arret eps=0.001")
plt.xlabel("nombre de features"); plt.ylabel("accuracy CV")
plt.title("Forward selection"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

Mon implémentation sélectionne les mêmes 10 variables que scikit-learn (seul l'ordre est différent). Le score monte vite (0,922 avec 1 variable, 0,982 avec 3) puis stagne vers 0,985–0,99. Avec ε = 0,001 l'algorithme s'arrête à 4 variables, parce que la 5e n'apporte aucun gain.

### C.2 Backward elimination

In [ ]:
t0 = time.perf_counter()
sfs_b = SequentialFeatureSelector(LogisticRegression(max_iter=1000), n_features_to_select=10,
                                  direction="backward", scoring="accuracy", cv=skf)
sfs_b.fit(Xtr_s, y_train)
t_sfs_b = time.perf_counter() - t0
feat_sfs_b = list(noms[sfs_b.get_support()])
print("Backward sklearn (%.1f s) :" % t_sfs_b, feat_sfs_b)

In [ ]:
def backward_elimination(Xa, ya, n_min=1):
    modele = LogisticRegression(max_iter=1000)
    restantes = list(range(Xa.shape[1]))
    s = cross_val_score(modele, Xa, ya, cv=skf, scoring="accuracy")
    historique = [(len(restantes), s.mean(), s.std(), list(restantes))]
    while len(restantes) > n_min:
        meilleur_score, meilleur_f, meilleur_std = -1, None, None
        for f in restantes:
            cols = [c for c in restantes if c != f]
            s = cross_val_score(modele, Xa[:, cols], ya, cv=skf, scoring="accuracy")
            if s.mean() > meilleur_score:
                meilleur_score, meilleur_f, meilleur_std = s.mean(), f, s.std()
        restantes.remove(meilleur_f)
        historique.append((len(restantes), meilleur_score, meilleur_std, list(restantes)))
    return historique

t0 = time.perf_counter()
hist_b = backward_elimination(Xtr_s, y_train)
t_backward_perso = time.perf_counter() - t0
feat_perso_b = [noms[i] for i in [h for h in hist_b if h[0] == 10][0][3]]
print("Backward perso (%.1f s) :" % t_backward_perso, feat_perso_b)
print("meme ensemble que sklearn ?", set(feat_perso_b) == set(feat_sfs_b))

plt.figure(figsize=(6, 4))
plt.errorbar([h[0] for h in hist_b], [h[1] for h in hist_b], yerr=[h[2] for h in hist_b],
             marker="o", capsize=2, ms=4)
plt.gca().invert_xaxis()
plt.xlabel("nombre de features restantes"); plt.ylabel("accuracy CV")
plt.title("Backward elimination"); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()
for h in hist_b:
    if h[0] <= 30 and h[0] % 3 == 0 or h[0] <= 10:
        print(h[0], round(h[1], 4))

In [ ]:
com = set(feat_sfs_f) & set(feat_sfs_b)
print("en commun (%d) :" % len(com), sorted(com))
print("seulement forward :", sorted(set(feat_sfs_f) - set(feat_sfs_b)))
print("seulement backward :", sorted(set(feat_sfs_b) - set(feat_sfs_f)))

Là aussi mon implémentation donne le même résultat que scikit-learn. L'accuracy CV est un peu meilleure autour de 15–21 variables (0,9899) qu'avec les 30 (0,9824), puis elle baisse quand il reste très peu de variables.

Forward et backward n'ont que 6 variables en commun sur 10. Ils ne sont pas identiques parce qu'ils ne partent pas du même endroit : le forward ajoute la variable qui aide le plus, le backward enlève celle qui gêne le moins. Comme beaucoup de variables sont redondantes, plusieurs choix sont presque équivalents, et une fois qu'un choix est fait il n'est jamais remis en cause.

### C.3 RFE

In [ ]:
rfe = RFE(LogisticRegression(max_iter=1000), n_features_to_select=10)
t0 = time.perf_counter()
rfe.fit(Xtr_s, y_train)
t_rfe = time.perf_counter() - t0
tab_rfe = pd.DataFrame({"support": rfe.support_, "ranking": rfe.ranking_}, index=noms).sort_values("ranking")
print("RFE (%.3f s)" % t_rfe)
print(tab_rfe)
feat_rfe = list(noms[rfe.support_])

In [ ]:
t0 = time.perf_counter()
rfecv = RFECV(LogisticRegression(max_iter=1000), cv=skf, scoring="accuracy", min_features_to_select=1)
rfecv.fit(Xtr_s, y_train)
t_rfecv = time.perf_counter() - t0
print("RFECV (%.2f s) nb optimal :" % t_rfecv, rfecv.n_features_)
feat_rfecv = list(noms[rfecv.support_])
print(feat_rfecv)

plt.figure(figsize=(6, 4))
nb = range(1, 31)
plt.errorbar(nb, rfecv.cv_results_["mean_test_score"], yerr=rfecv.cv_results_["std_test_score"],
             marker="o", ms=3, capsize=2)
plt.axvline(rfecv.n_features_, color="red", ls="--", label="optimum = %d" % rfecv.n_features_)
plt.xlabel("nombre de features"); plt.ylabel("accuracy CV")
plt.title("RFECV (regression logistique)"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

cvm = rfecv.cv_results_["mean_test_score"]; cvs = rfecv.cv_results_["std_test_score"]
ib = cvm.argmax()
n_rfecv_1std = min(n for n, m in zip(nb, cvm) if m >= cvm[ib] - cvs[ib])
print("meilleur :", ib + 1, round(cvm[ib], 4), "| plus petit dans 1 std :", n_rfecv_1std)

In [ ]:
rfe_svc = RFE(SVC(kernel="linear"), n_features_to_select=10).fit(Xtr_s, y_train)
rfe_rf = RFE(RandomForestClassifier(n_estimators=200, random_state=SEED), n_features_to_select=10).fit(Xtr_s, y_train)
f_svc = set(noms[rfe_svc.support_]); f_rf = set(noms[rfe_rf.support_])
print("SVC :", sorted(f_svc))
print("RF  :", sorted(f_rf))
print("LR & SVC :", len(set(feat_rfe) & f_svc), "| LR & RF :", len(set(feat_rfe) & f_rf), "| SVC & RF :", len(f_svc & f_rf))

RFE garde 10 variables (mean area, mean concave points, area error, worst concave points, worst radius, worst texture, worst perimeter, worst area, worst symmetry, worst smoothness). Les premières éliminées sont worst compactness, mean fractal dimension et mean symmetry. RFECV donne un nombre optimal de 8 variables.

Avec SVC linéaire 8 variables sur 10 sont les mêmes que la régression logistique, et avec la random forest seulement 6. Ce n'est pas surprenant : chaque modèle a son propre critère (coefficients pour LR et SVC, importance pour la forêt), et avec des variables corrélées ils ne répartissent pas l'importance de la même façon.

Différence entre RFE et backward : RFE enlève la variable avec le plus petit coefficient du modèle, sans validation croisée, et il entraîne un seul modèle par itération (20 modèles pour passer de 30 à 10). Le backward teste la suppression de chaque variable avec la validation croisée : (30 + 29 + … + 11) × 5 = 2050 modèles.

### C.4 Coût

In [ ]:
p, k, cv = 30, 10, 5
nb_eval_forward = sum(p - i for i in range(k))
print("forward : %d sous-ensembles evalues x %d plis = %d modeles" % (nb_eval_forward, cv, nb_eval_forward * cv))
nb_subsets = 2 ** p - 1
print("recherche exhaustive : %d sous-ensembles x %d = %.2e modeles" % (nb_subsets, cv, nb_subsets * cv))
print("seulement les sous-ensembles de taille 10 : C(30,10) =", __import__('math').comb(30, 10), "->", __import__('math').comb(30, 10) * cv, "modeles")

# temps moyen d'un modele
t0 = time.perf_counter()
for i in range(50):
    LogisticRegression(max_iter=1000).fit(Xtr_s[:, :10], y_train)
t_un = (time.perf_counter() - t0) / 50
print("temps moyen d'un fit : %.4f s" % t_un)
print("estimation forward : %.1f s | mesure : %.1f s" % (nb_eval_forward * cv * t_un, t_forward_perso))
print("estimation exhaustive : %.1f jours" % (nb_subsets * cv * t_un / 86400))

- Forward, p = 30, k = 10 : 30 + 29 + … + 21 = 255 sous-ensembles, donc 255 × 5 = 1275 modèles.
- Recherche exhaustive : 2³⁰ − 1 ≈ 1,07 milliard de sous-ensembles, donc environ 5,4 × 10⁹ modèles (≈ 143 jours à 2,3 ms par modèle).
- Estimation du forward : 1275 × 2,3 ms ≈ 2,9 s, et j'ai mesuré environ 5 s. Le temps réel est plus grand parce que `cross_val_score` ajoute du temps en plus de l'entraînement, mais l'ordre de grandeur est correct.

## Partie D – Bruit, fuite et comparaison finale

### D.1 Bruit

In [ ]:
rng = np.random.default_rng(SEED)
noise_cols = ["noise_%02d" % i for i in range(20)]
noise_train = pd.DataFrame(rng.standard_normal((len(X_train), 20)), columns=noise_cols, index=X_train.index)
noise_test = pd.DataFrame(rng.standard_normal((len(X_test), 20)), columns=noise_cols, index=X_test.index)
X_train50 = pd.concat([X_train, noise_train], axis=1)
X_test50 = pd.concat([X_test, noise_test], axis=1)
print(X_train50.shape, X_test50.shape)

Xtr50_s = StandardScaler().fit_transform(X_train50)
noms50 = np.array(X_train50.columns)

def compter_bruit(masque):
    choisies = noms50[masque]
    n = sum(c.startswith("noise") for c in choisies)
    return n, list(choisies)

res_bruit = {}
kb = SelectKBest(f_classif, k=10).fit(Xtr50_s, y_train)
res_bruit["SelectKBest (F)"] = compter_bruit(kb.get_support())

fw = SequentialFeatureSelector(LogisticRegression(max_iter=1000), n_features_to_select=10,
                               direction="forward", scoring="accuracy", cv=skf).fit(Xtr50_s, y_train)
res_bruit["Forward"] = compter_bruit(fw.get_support())

bw = SequentialFeatureSelector(LogisticRegression(max_iter=1000), n_features_to_select=10,
                               direction="backward", scoring="accuracy", cv=skf).fit(Xtr50_s, y_train)
res_bruit["Backward"] = compter_bruit(bw.get_support())

rf = RFE(LogisticRegression(max_iter=1000), n_features_to_select=10).fit(Xtr50_s, y_train)
res_bruit["RFE"] = compter_bruit(rf.get_support())

for nom in res_bruit:
    print(nom, ":", res_bruit[nom][0], "bruit sur 10 ->", res_bruit[nom][1])

| Méthode | variables de bruit sur 10 |
|---|---|
| SelectKBest (F) | 0 |
| Forward | 0 |
| RFE | 0 |
| Backward | 5 |

Le backward est le moins robuste. Le test F donne un score proche de 0 au bruit donc il n'est jamais gardé, et forward et RFE partent des bonnes variables. Le backward part des 50 variables et enlève celle qui fait le moins baisser le score de validation croisée : comme beaucoup de variables utiles sont redondantes, on peut les enlever sans perdre beaucoup, et le bruit peut rester par hasard. Les wrappers sont plus à risque d'être trompés parce qu'ils optimisent un score de CV qui est lui-même un peu bruité.

### D.2 Fuite de données

In [ ]:
# incorrect : on selectionne sur tout le train, puis on fait la CV
sel = SelectKBest(f_classif, k=10).fit(Xtr50_s, y_train)
X_red = sel.transform(Xtr50_s)
acc_incorrect = cross_val_score(LogisticRegression(max_iter=1000), X_red, y_train, cv=skf)

# correct : tout dans un pipeline
pipe = Pipeline([("sc", StandardScaler()), ("sel", SelectKBest(f_classif, k=10)),
                 ("lr", LogisticRegression(max_iter=1000))])
acc_correct = cross_val_score(pipe, X_train50, y_train, cv=skf)
print("incorrect :", round(acc_incorrect.mean(), 4), "+/-", round(acc_incorrect.std(), 4))
print("correct   :", round(acc_correct.mean(), 4), "+/-", round(acc_correct.std(), 4))

Version incorrecte : 0,9548. Version correcte (Pipeline) : 0,9523. L'écart est très petit.

Dans la version incorrecte, le test F utilise la cible de toutes les observations du train, y compris celles qui servent ensuite de validation, donc l'information des plis de validation fuit dans le choix des variables. Ici l'écart est petit parce qu'il y a beaucoup de vraies variables informatives : le test F choisit presque les mêmes variables quel que soit le pli. L'écart devient grand quand on a peu d'observations et beaucoup de variables, parce que la sélection peut alors choisir par hasard des variables qui « ressemblent » à la cible sur ces données.

### D.3 Comparaison finale sur le test

In [ ]:
# nombre de variables choisi avec les courbes de validation (C.1, C.2, RFECV)
n_fwd = len(hist_f)      # arret avec eps = 0.001
n_bwd = 10
print("k F =", best_k["F"], "| k MI =", best_k["MI"], "| forward =", n_fwd, "| backward =", n_bwd)

def faire_pipe(selecteur):
    return Pipeline([("sc", StandardScaler()), ("sel", selecteur), ("lr", LogisticRegression(max_iter=1000))])

lignes = [{"Methode": "Baseline (toutes)", "Nb features": 30, "Test acc": baseline["Test acc"][0],
           "Test F1": baseline["Test F1"][0], "Temps (s)": np.nan}]
supports = {}

# filtre correlation (seuil 0.95 choisi en B.2) : on selectionne sur le train seulement
t0 = time.perf_counter()
kept = remove_correlated(X_train, y_train, 0.95)
p_corr = Pipeline([("sc", StandardScaler()), ("lr", LogisticRegression(max_iter=1000))])
p_corr.fit(X_train[kept], y_train)
duree = time.perf_counter() - t0
pred = p_corr.predict(X_test[kept])
supports["Correlation"] = np.array([c in kept for c in X_train.columns])
lignes.append({"Methode": "Filtre : correlation (B.2)", "Nb features": len(kept),
               "Test acc": accuracy_score(y_test, pred), "Test F1": f1_score(y_test, pred), "Temps (s)": duree})

autres = {
    "SelectKBest F": faire_pipe(SelectKBest(f_classif, k=best_k["F"])),
    "SelectKBest MI": faire_pipe(SelectKBest(mi_func, k=best_k["MI"])),
    "Forward": faire_pipe(SequentialFeatureSelector(LogisticRegression(max_iter=1000),
                 n_features_to_select=n_fwd, direction="forward", scoring="accuracy", cv=skf)),
    "Backward": faire_pipe(SequentialFeatureSelector(LogisticRegression(max_iter=1000),
                 n_features_to_select=n_bwd, direction="backward", scoring="accuracy", cv=skf)),
    "RFE": faire_pipe(RFE(LogisticRegression(max_iter=1000), n_features_to_select=10)),
    "RFECV": faire_pipe(RFECV(LogisticRegression(max_iter=1000), cv=skf, scoring="accuracy")),
}
for nom in autres:
    pipe = autres[nom]
    t0 = time.perf_counter()
    pipe.fit(X_train, y_train)
    duree = time.perf_counter() - t0
    pred = pipe.predict(X_test)
    masque = pipe.named_steps["sel"].get_support()
    supports[nom] = masque
    lignes.append({"Methode": nom, "Nb features": int(masque.sum()),
                   "Test acc": accuracy_score(y_test, pred), "Test F1": f1_score(y_test, pred), "Temps (s)": duree})

final = pd.DataFrame(lignes)
print(final.round(4).to_string(index=False))

# matrice de presence
presence = pd.DataFrame(supports, index=X_train.columns).astype(int)
presence["total"] = presence.sum(axis=1)
presence = presence.sort_values("total", ascending=False)
print(presence.to_string())
print("choisies par toutes les methodes :", list(presence.index[presence["total"] == 7]))
print("choisies par au moins 5 methodes :", list(presence.index[presence["total"] >= 5]))

Paramètres choisis sur la validation : seuil de corrélation 0,95, k = 20 pour SelectKBest, 4 variables pour le forward (critère ε de C.1), 10 pour le backward et RFE, 8 pour RFECV. Pour le filtre de corrélation, la sélection est faite sur le train seulement, puis le modèle est évalué sur le test.

| Méthode | Nb variables | Accuracy test | F1 test | Temps (s) |
|---|---|---|---|---|
| Baseline | 30 | 0,9883 | 0,9907 | – |
| Filtre : corrélation | 23 | 0,9825 | 0,9860 | 0,01 |
| SelectKBest F | 20 | 0,9766 | 0,9813 | 0,006 |
| SelectKBest MI | 20 | 0,9708 | 0,9770 | 0,07 |
| Forward | 4 | 0,9415 | 0,9528 | 2,1 |
| Backward | 10 | 0,9591 | 0,9677 | 9,0 |
| RFE | 10 | 0,9708 | 0,9767 | 0,05 |
| RFECV | 8 | 0,9591 | 0,9671 | 0,6 |

(les temps changent un peu à chaque exécution)

Une seule variable est choisie par toutes les méthodes : worst texture. Sept variables sont choisies par au moins 5 méthodes sur 7 : worst texture, area error, worst perimeter, worst smoothness, worst radius, worst concave points et worst symmetry.

C'est cohérent avec la partie A : les méthodes ne gardent pas toutes la même variable du groupe radius/perimeter/area (elles sont presque interchangeables), mais elles en gardent au moins une. Worst texture et worst smoothness sont peu corrélées avec ce groupe (0,36 et 0,22 avec worst radius), donc elles apportent une information différente et sont gardées presque partout.

## Discussion finale

**1. Meilleure accuracy avec le moins de variables ?** Aucune méthode ne dépasse la baseline (0,9883). RFE avec 10 variables (0,9708) est un bon compromis. L'écart avec la baseline est de 3 patients sur 171 (un patient = 0,58 point), ce qui n'est pas significatif avec un si petit test. Pour le forward avec 4 variables (0,9415) la perte est de 8 patients, donc plus nette.

**2. Filtres contre wrappers.** Les filtres sont très rapides et ne dépendent pas du modèle, mais ils notent chaque variable séparément : ils ne gèrent pas la redondance ni les interactions. Les wrappers utilisent le vrai modèle donc ils gèrent mieux la redondance, mais ils sont plus lents et dépendent du modèle choisi.

**3. Pourquoi les wrappers coûtent plus cher ?** Il faut entraîner un modèle pour chaque sous-ensemble candidat et chaque pli. Pour le forward avec p variables et k sélectionnées, avec 5 plis : 5 × (p + (p−1) + … + (p−k+1)) modèles, soit 1275 pour p = 30 et k = 10.

**4. Forward et backward garantissent-ils l'optimum ?** Non. Mes deux méthodes donnent des ensembles différents (6 variables en commun sur 10) avec des scores CV différents (0,9899 pour le forward et 0,9849 pour le backward à 10 variables), donc au moins l'un des deux n'est pas optimal. Ils sont gloutons : un choix fait n'est jamais remis en question.

**5. Les variables de bruit.** SelectKBest, forward et RFE n'ont gardé aucun bruit, alors que le backward en a gardé 5 sur 10. Les filtres sont robustes ici parce que le bruit a un score F presque nul. Les wrappers peuvent se faire tromper parce que le score de CV varie un peu.

**6. Pourquoi la sélection doit être dans la validation croisée ?** Parce que la sélection utilise la cible : si on la fait avant la CV, les données de validation ont déjà servi à choisir les variables et le score est trop optimiste. Le Pipeline refait le scaler et la sélection à chaque pli avec seulement les données d'entraînement du pli.

**7. Quelle famille selon la situation ?**
- Des milliers de variables : filtre, parce que c'est rapide.
- Petit jeu de données : sélection dans la validation croisée avec prudence (risque de fuite et de surapprentissage).
- Besoin d'interprétabilité : RFE ou forward/backward pour avoir peu de variables.
- Temps de calcul limité : filtre, ou RFE.